In [ ]:

import numpy as np
import pandas as pd

import os
!pip install -q transformers datasets sentencepiece accelerate

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#loading tokenizer
from transformers import AutoTokenizer
import torch

MODEL_NAME = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Model will be loaded from checkpoint in Cell 8
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
# model = model.to("cuda")


In [ ]:
from datasets import Dataset, DatasetDict
import json, os

DATA_ROOT = "/kaggle/input/datasets/jlkeyx/cultural-recipes-nlp"

def load_flat_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

raw_datasets = {}
for direction in ["cn2en", "en2cn"]:
    train = load_flat_jsonl(f"{DATA_ROOT}/recipe_match_{direction}_train.jsonl")
    valid = load_flat_jsonl(f"{DATA_ROOT}/recipe_match_{direction}_valid.jsonl")
    test  = load_flat_jsonl(f"{DATA_ROOT}/recipe_match_{direction}_test.jsonl")

    raw_datasets[f"recipe_match_{direction}"] = DatasetDict({
        "train": Dataset.from_list(train),
        "valid": Dataset.from_list(valid),
        "test":  Dataset.from_list(test),
    })
    print(f"recipe_match_{direction}: train={len(train):,}, "
          f"valid={len(valid):,}, test={len(test):,}")


In [ ]:
#tokenization
MAX_INPUT_LENGTH  = 512
MAX_TARGET_LENGTH = 512

def tokenize(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = {}
for direction in ["recipe_match_cn2en", "recipe_match_en2cn"]:
    tokenized[direction] = raw_datasets[direction].map(
        tokenize,
        batched=True,
        batch_size=64,
        remove_columns=["source", "target"],
    )


In [ ]:
#combine en2cn and cn2en
from datasets import concatenate_datasets

combined = DatasetDict({
    "train": concatenate_datasets([
        tokenized["recipe_match_cn2en"]["train"],
        tokenized["recipe_match_en2cn"]["train"],
    ]),
    "valid": concatenate_datasets([
        tokenized["recipe_match_cn2en"]["valid"],
        tokenized["recipe_match_en2cn"]["valid"],
    ]),
    "test": concatenate_datasets([
        tokenized["recipe_match_cn2en"]["test"],
        tokenized["recipe_match_en2cn"]["test"],
    ]),
})

print(f"Combined dataset:")
print(f"  train: {len(combined['train']):,} pairs")
print(f"  valid: {len(combined['valid']):,} pairs")
print(f"  test:  {len(combined['test']):,} pairs")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import (Seq2SeqTrainingArguments,
                          Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)

OUTPUT_DIR = "/kaggle/working/cultural-recipes-model"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=5e-4,
    label_smoothing_factor=0.0,
    predict_with_generate=True,
    generation_max_length=128,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=False,
    fp16=False,
    logging_steps=50,
    save_total_limit=3,
    report_to="none",
    gradient_checkpointing=True,
    optim="adafactor",
)

steps_per_epoch = len(combined["train"]) // (
    training_args.per_device_train_batch_size *
    training_args.gradient_accumulation_steps
)
est_hours = (steps_per_epoch * 3.5) / 3600  # estimate for 1 epoch only

print(f"  Estimated time:   ~{est_hours:.1f} hours for epoch")

In [ ]:
#loading checkpoint
import torch, gc, os, shutil
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, DataCollatorForSeq2Seq
from torch.utils.data import DataLoader

CHECKPOINT_BASE = "/kaggle/input/datasets/jlkeyx/cultural-recipes-checkpoint/cultural-recipes-model"
RESUME_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "checkpoint-13500")


print("\nLoading model from checkpoint-13500...")
model = AutoModelForSeq2SeqLM.from_pretrained(RESUME_CHECKPOINT)
model = model.to("cuda")
param_sum = sum(p.sum().item() for p in model.parameters())
print(f"Model loaded! Parameter checksum: {param_sum:.2f}")

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8)
print("Data collator created")

loader = DataLoader(
    combined["train"].select(range(4)),
    batch_size=2, collate_fn=data_collator)
model.train()
batch = next(iter(loader))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print(f"Sanity check loss: {out.loss.item():.4f}")
print(f"(Should be well below 25 if checkpoint loaded correctly)")
assert not torch.isnan(out.loss), "NaN loss — stopping!"

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=combined["train"],
    eval_dataset=combined["valid"],
    processing_class=tokenizer,
    data_collator=data_collator,
)
print("Trainer created with checkpoint model")

gc.collect()
torch.cuda.empty_cache()
used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {used:.1f} / {total:.1f} GB | Free: {total-used:.1f} GB")

print("\nResuming training from checkpoint-13500...")
trainer.train(
    resume_from_checkpoint=os.path.join(OUTPUT_DIR, "checkpoint-13500")
)

trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print("\nTraining complete! Model saved.")

print("\nSaved files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1024 / 1024
        if size > 0.1:
            print(f"  {fpath}  ({size:.1f} MB)")

In [ ]:
model.eval()

def adapt_recipe(recipe_text, max_length=256):
    inputs = tokenizer(
        recipe_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=4,
        repetition_penalty=2.5,
        length_penalty=1.0,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

en_recipe = (
    "Title: Scrambled Eggs, "
    "Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper, "
    "Steps: Crack eggs into bowl. Whisk well. "
    "Melt butter in pan over medium heat. "
    "Add eggs and stir gently until cooked."
)

cn_recipe = (
    "Title: 番茄炒蛋, "
    "Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, "
    "Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，"
    "加入番茄翻炒，加盐和糖调味即可。"
)

print("=== TEST 1: English → Chinese ===")
print("INPUT: ", en_recipe[:80], "...")
print("OUTPUT:", adapt_recipe(en_recipe))

print("\n=== TEST 2: Chinese → English ===")
print("INPUT: ", cn_recipe[:80], "...")
print("OUTPUT:", adapt_recipe(cn_recipe))

In [ ]:
#saving checkpoint for next session and google colab import

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1024 / 1024
        if size > 0.1:  # only show files > 0.1MB
            print(f"  {fpath}  ({size:.1f} MB)")